In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from arch import arch_model
import os
from scipy.stats import chi2
from scipy.stats import skew, kurtosis
from statsmodels.tsa.stattools import acovf 

In [ ]:
# Read data
data = pd.read_excel('Assignment3Data.xlsx', sheet_name='GoyalWelch')

# Question 1

In [ ]:
# Read data
data = pd.read_excel('Assignment3Data.xlsx', sheet_name='GoyalWelch')

# Excess returns
data['Excess_Return'] =(data['Return'] - data['T-Bill']) * 100

# Shift to get rt+1
data['Excess_Return_t+1'] = data['Excess_Return'].shift(-1)

# Lagged return rt
data['Lag_Excess_Return'] = data['Excess_Return']

# E/P ratio
data['E/P'] = data['E12'] / data['Index']

data = data.dropna(subset=['Excess_Return_t+1', 'Lag_Excess_Return', 'E/P'])

# # Regression with Lagged Return
X1 = data['Lag_Excess_Return']
y = data['Excess_Return_t+1']
X1 = sm.add_constant(X1)
model1 = sm.OLS(y, X1).fit()
print('Regression with Lagged Return:')
print(model1.summary())

# Regression with E/P Ratio
X2 = data['E/P']
X2 = sm.add_constant(X2)
model2 = sm.OLS(y, X2).fit()
print('Regression with E/P Ratio:')
print(model2.summary())

In [ ]:
# 1b)

# Read the data
data = pd.read_excel('Assignment3Data.xlsx', sheet_name='GoyalWelch')

# Prepare data
data['Excess_Return'] = data['Return'] - data['T-Bill']
data['Excess_Return_t+1'] = data['Excess_Return'].shift(-1)
data['Lag_Excess_Return'] = data['Excess_Return']
data['E/P'] = data['E12'] / data['Index']
data = data.dropna(subset=['Excess_Return_t+1', 'Lag_Excess_Return', 'E/P'])

# Lists to store data
results = []
start_year = 1946
initial_end_year = 1970
years = data['Year'].unique()
years.sort()

for end_year in range(initial_end_year + 1, max(years) + 1):
    # Training data: up to end_year -1 and test data
    train_data = data[(data['Year'] >= start_year) & (data['Year'] <= end_year - 1)]
    test_data = data[data['Year'] == end_year]
    
    # Actual return for t+1
    actual_return = test_data['Excess_Return_t+1'].values[0]
    
    # Historical Average
    historical_mean = train_data['Excess_Return_t+1'].mean()
    
    # AR(1) Regression
    X_train_ar1 = sm.add_constant(train_data['Lag_Excess_Return'])
    model_ar1 = sm.OLS(train_data['Excess_Return_t+1'], X_train_ar1).fit()
    
    # E/P Regression
    X_train_ep = sm.add_constant(train_data['E/P'])
    model_ep = sm.OLS(train_data['Excess_Return_t+1'], X_train_ep).fit()
    
    # Predictions
    
    # AR(1) Prediction
    X_test_ar1 = pd.DataFrame({
        'const': 1,
        'Lag_Excess_Return': test_data['Lag_Excess_Return'].values[0]
    }, index=[0])
    
    pred_ar1 = model_ar1.predict(X_test_ar1)[0]
    
    # E/P Prediction
    X_test_ep = pd.DataFrame({
        'const': 1,
        'E/P': test_data['E/P'].values[0]
    }, index=[0])
    
    pred_ep = model_ep.predict(X_test_ep)[0]
    
    # Combined Forecast (average of AR(1) and E/P predictions)
    pred_combined = (pred_ar1 + pred_ep) / 2
    
    # Store Results
    results.append({
        'Year': end_year,
        'Actual': actual_return,
        'Forecast_Hist': historical_mean,
        'Forecast_AR1': pred_ar1,
        'Forecast_EP': pred_ep,
        'Forecast_Combined': pred_combined
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Calculate Errors
for model in ['Hist', 'AR1', 'EP', 'Combined']:
    results_df[f'Error_{model}'] = results_df['Actual'] - results_df[f'Forecast_{model}']

# Calculate MSPE
mspe = {}
for model in ['Hist', 'AR1', 'EP', 'Combined']:
    mspe[model] = np.mean(results_df[f'Error_{model}'] ** 2)
print('MSPE:', mspe)

# Calculate Out-of-Sample R-squared
def compute_y_bar(row):
    train_year = row['Year'] - 1
    relevant_train = data[(data['Year'] >= start_year) & (data['Year'] <= train_year)]
    return relevant_train['Excess_Return_t+1'].mean()

results_df['Y_bar'] = results_df.apply(compute_y_bar, axis=1)

denominator = np.sum((results_df['Actual'] - results_df['Y_bar']) ** 2)
r2_os = {}
for model in ['AR1', 'EP', 'Combined']:
    numerator = np.sum(results_df[f'Error_{model}'] ** 2)
    r2_os[model] = 1 - (numerator / denominator)
print('Out-of-Sample R-squared:', r2_os)

# Plot Cumulative MSPE
for model in ['Hist', 'AR1', 'EP', 'Combined']:
    results_df[f'Cumulative_MSPE_{model}'] = np.cumsum(results_df[f'Error_{model}'] ** 2)

plt.figure(figsize=(10,6))
for model in ['Hist', 'AR1', 'EP', 'Combined']:
    plt.plot(results_df['Year'], results_df[f'Cumulative_MSPE_{model}'], label=model)
plt.xlabel('Year')
plt.ylabel('Cumulative MSPE')
plt.legend()
plt.title('Cumulative MSPE over Time')
plt.grid(True)
plt.savefig('A2_2b_predictions.png', dpi=300)
plt.show()



In [ ]:
# 1c)
from statsmodels.tsa.stattools import acovf 

# Calculate squared forecast errors
for model in ['AR1', 'EP', 'Combined']:
    results_df[f'd_{model}'] = results_df['Error_Hist'] ** 2 - results_df[f'Error_{model}'] ** 2

# Diebold-Mariano test function
def compute_dm_test(loss_diff):
    T = len(loss_diff)
    mean_loss_diff = np.mean(loss_diff)
    var_loss_diff = acovf(loss_diff, fft=False, nlag=1, demean=True)  # Compute autocovariances
    variance = (var_loss_diff[0] + 2 * var_loss_diff[1]) / T  # DM test variance calculation
    se = np.sqrt(variance)  # Standard error
    t_stat = mean_loss_diff / se  # t-statistic
    return t_stat

# Clark-West adjustment function
def clark_west_adjustment(actual, forecast_h0, forecast_h1):
    fhat_diff = forecast_h1 - forecast_h0
    adjustment = fhat_diff ** 2
    loss_diff = (actual - forecast_h0) ** 2 - (actual - forecast_h1) ** 2 + adjustment
    return loss_diff

# Diebold-Mariano t-statistic
for model in ['AR1', 'EP', 'Combined']:
    t_stat = compute_dm_test(results_df[f'd_{model}'])
    print(f'Diebold-Mariano t-statistic for {model}: {t_stat}')

# Clark-West t-statistic
for model in ['AR1', 'EP', 'Combined']:
    loss_diff_cw = clark_west_adjustment(results_df['Actual'], results_df['Forecast_Hist'], results_df[f'Forecast_{model}'])
    t_stat_cw = compute_dm_test(loss_diff_cw)
    print(f'Clark-West t-statistic for {model}: {t_stat_cw}')


In [ ]:
# 1d) Bootstrap for the E/P model

np.random.seed(222)

num_bootstrap = 5000  # Number of bootstrap iterations
t_stats_dm_bootstrap = np.zeros(num_bootstrap)
t_stats_cw_bootstrap = np.zeros(num_bootstrap)

# Get years and returns
test_years = results_df['Year'].values
returns_sample = data[data['Year'].isin(test_years)]['Excess_Return_t+1'].values

for i in range(num_bootstrap):
    boot_returns = np.random.choice(returns_sample, size=len(returns_sample), replace=True)

    # Calculate errors
    boot_errors_hist = boot_returns - results_df['Forecast_Hist']
    boot_errors_ep = boot_returns - results_df['Forecast_EP']
    boot_d_ep = boot_errors_hist ** 2 - boot_errors_ep ** 2

    # DM
    try:
        t_stats_dm_bootstrap[i] = compute_dm_test(boot_d_ep)
    except ZeroDivisionError:
        t_stats_dm_bootstrap[i] = np.nan  # Zero check, got error sometimes. 

    # CW
    boot_loss_diff_cw_ep = clark_west_adjustment(boot_returns, results_df['Forecast_Hist'], results_df['Forecast_EP'])
    try:
        t_stats_cw_bootstrap[i] = compute_dm_test(boot_loss_diff_cw_ep)
    except ZeroDivisionError:
        t_stats_cw_bootstrap[i] = np.nan 

# Remove na
t_stats_dm_bootstrap = t_stats_dm_bootstrap[~np.isnan(t_stats_dm_bootstrap)]
t_stats_cw_bootstrap = t_stats_cw_bootstrap[~np.isnan(t_stats_cw_bootstrap)]

# 5%-95% CI
dm_percentiles = np.percentile(t_stats_dm_bootstrap, [5, 95])
cw_percentiles = np.percentile(t_stats_cw_bootstrap, [5, 95])
print('DM t-statistic percentiles:', dm_percentiles)
print('CW t-statistic percentiles:', cw_percentiles)

# Means
print('DM t-statistic mean:', np.mean(t_stats_dm_bootstrap))
print('CW t-statistic mean:', np.mean(t_stats_cw_bootstrap))


# Plot histograms for visualization of symmetry
plt.figure(figsize=(12, 6))

# Histogram for Diebold-Mariano t-statistics
plt.subplot(1, 2, 1)
plt.hist(t_stats_dm_bootstrap, bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.axvline(np.mean(t_stats_dm_bootstrap), color='red', linestyle='--', label='Mean')
plt.title('Bootstrap Distribution of DM t-statistics')
plt.xlabel('t-statistic')
plt.ylabel('Frequency')
plt.legend()

# Histogram for Clark-West t-statistics
plt.subplot(1, 2, 2)
plt.hist(t_stats_cw_bootstrap, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.axvline(np.mean(t_stats_cw_bootstrap), color='red', linestyle='--', label='Mean')
plt.title('Bootstrap Distribution of CW t-statistics')
plt.xlabel('t-statistic')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.savefig('A3_2d_histogram.jpg', format='jpg', dpi=300)
plt.show()

# Skewness and kurtosis for DM t-statistics
dm_skewness = skew(t_stats_dm_bootstrap)
dm_kurtosis = kurtosis(t_stats_dm_bootstrap)
print(f'DM Skewness: {dm_skewness:.4f}, DM Kurtosis: {dm_kurtosis:.4f}')

# Skewness and kurtosis for CW t-statistics
cw_skewness = skew(t_stats_cw_bootstrap)
cw_kurtosis = kurtosis(t_stats_cw_bootstrap)
print(f'CW Skewness: {cw_skewness:.4f}, CW Kurtosis: {cw_kurtosis:.4f}')

# Question 3

In [ ]:

# 3a)

sp500_data = pd.read_excel('Assignment3Data.xlsx', sheet_name='S&P500')
sp500_data['Date'] = pd.to_datetime(sp500_data[['Year', 'Month', 'Day']])
sp500_data.set_index('Date', inplace=True)

# Convert to log returns
sp500_data['Log_Return'] = np.log(1 + sp500_data['Return'] / 100)
sp500_data.dropna(inplace=True)

# Rescale
sp500_data['Log_Return_Scaled'] = sp500_data['Log_Return'] * 100

# ARCH(1)
am_arch = arch_model(sp500_data['Log_Return_Scaled'], mean='constant', vol='ARCH', p=1, dist='normal')

res_arch = am_arch.fit(update_freq=5, disp='off')
print('ARCH(1) Model Summary (Rescaled Data):')
print(res_arch.summary())

# Standard Errors (Inverse Hessian)
print('Standard Errors (Inverse Hessian):')
print(res_arch.param_cov)

# Quasi Maximum Likelihood Estimation (QMLE)
res_arch_qmle = am_arch.fit(update_freq=5, disp='off', cov_type='robust')
print('ARCH(1) Model Summary (QMLE):')
print(res_arch_qmle.summary())
print('Standard Errors (QMLE):')
print(res_arch_qmle.param_cov)

plt.figure(figsize=(10,6))
plt.plot(res_arch.conditional_volatility * np.sqrt(252))
plt.title('Conditional Volatility from ARCH(1)')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility')
plt.show()


In [ ]:
# 3b)

# GARCH(1, 1)
am_garch = arch_model(sp500_data['Log_Return_Scaled'], mean='constant', vol='GARCH', p=1, q=1, dist='normal')
res_garch = am_garch.fit(update_freq=5, disp='off', cov_type='robust')
print(res_garch.summary())

plt.figure(figsize=(10,6))
plt.plot(res_garch.conditional_volatility)
plt.title('Conditional Volatility from GARCH(1,1)')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility')
plt.savefig('A3_3b_garch.jpg', format='jpg', dpi=300)
plt.show()

In [ ]:

# likliehood ratio of ARCH vs GARCH
ll_arch = res_arch.loglikelihood
ll_garch = res_garch.loglikelihood
LR_stat = 2 * (ll_garch - ll_arch)
p_value = 1 - chi2.cdf(LR_stat, df=1)
print('Likelihood Ratio Test Statistic:', LR_stat)
print('p-value:', p_value)

In [ ]:
# 3c)

# Calculate EWMA for 3 different lambdas
def compute_ewma_volatility(returns, lam):
    T = len(returns)
    sigma2 = np.zeros(T)
    sigma2[0] = np.var(returns)
    for t in range(1, T):
        sigma2[t] = (1 - lam) * returns[t-1] ** 2 + lam * sigma2[t-1]
    return np.sqrt(sigma2)

lambdas = [0.5, 0.9, 0.99]
plt.figure(figsize=(10,6))
for lam in lambdas:
    ewma_vol = compute_ewma_volatility(sp500_data['Log_Return'].values, lam)
    plt.plot(sp500_data.index, ewma_vol * np.sqrt(252), label=f'λ={lam}')
plt.title('EWMA Conditional Volatility')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility')
plt.legend()
plt.savefig('A3_3c_ewma.jpg', format='jpg', dpi=300)
plt.show()


In [ ]:
# 3d)

# GJR-GARCH
am_gjr = arch_model(sp500_data['Log_Return_Scaled'], mean='constant', vol='GARCH', p=1, o=1, q=1, dist='normal')
res_gjr = am_gjr.fit(update_freq=5, disp='off', cov_type='robust')
print(res_gjr.summary())

# Likelihood ratio test of GARCH vs GJR-GARCH
ll_garch = res_garch.loglikelihood
ll_gjr = res_gjr.loglikelihood
LR_stat = 2 * (ll_gjr - ll_garch)
p_value = 1 - chi2.cdf(LR_stat, df=1)
print('Likelihood GARCH:', (ll_garch / len(sp500_data['Log_Return_Scaled'])))
print('Likelihood GJR:', (ll_gjr / len(sp500_data['Log_Return_Scaled'])))
print('Likelihood Ratio Test Statistic:', LR_stat)
print('p-value:', p_value)

In [ ]:
# Parameters from GARCH(1,1)
alpha0 = res_garch.params['omega']
alpha1 = res_garch.params['alpha[1]']
beta1 = res_garch.params['beta[1]']
sigma2_bar = res_garch.conditional_volatility.mean() ** 2

# Parameters from GJR-GARCH(1,1)
alpha0_gjr = res_gjr.params['omega']
alpha1_gjr = res_gjr.params['alpha[1]']
beta1_gjr = res_gjr.params['beta[1]']
gamma1_gjr = res_gjr.params['gamma[1]']

# Range for epsilon
epsilon_range = np.linspace(-10, 10, 100)

# GARCH(1,1) News Impact Curve
sigma_garch = np.sqrt(alpha0 + alpha1 * epsilon_range ** 2 + beta1 * sigma2_bar)

# GJR-GARCH(1,1) News Impact Curve
D = (epsilon_range < 0).astype(float)
sigma_gjr = np.sqrt(alpha0_gjr + alpha1_gjr * epsilon_range ** 2 + beta1_gjr * sigma2_bar + gamma1_gjr * D * epsilon_range ** 2)

# Plot
plt.figure(figsize=(10,6))
plt.plot(epsilon_range, sigma_garch, label='GARCH(1,1)')
plt.plot(epsilon_range, sigma_gjr, label='GJR-GARCH(1,1)')
plt.title('News Impact Curves')
plt.xlabel('Lagged εₜ₋₁')
plt.ylabel('Conditional Standard Deviation')
plt.legend()
plt.savefig('A3_3d_newsimpactcurve.jpg', format='jpg', dpi=300)
plt.show()
